## Pre-trained models and Transfer Learning
Can we use a neural network trained on one dataset and adapt it to classifying different images without full training process?

This approach is called **Transfer Learning** 

In [ ]:
import tensorflow as tf
from tensorflow import keras
import matplotlib.pyplot as plt
import numpy as np
import os
from tfcv import *

from pytorchcv import check_image_dir

### Loading dataset

In [ ]:
if not os.path.exists('data/kagglecatsanddogs_5340.zip'):
    !wget -P data https://download.microsoft.com/download/3/E/1/3E1C3F21-ECDB-4869-8368-6DEBA77B919F/kagglecatsanddogs_5340.zip

import zipfile
if not os.path.exists('data/pet_images'):
    with zipfile.ZipFile('data/kagglecatsanddogs_5340.zip', 'r') as zip_ref:
        zip_ref.extractall('data')

check_image_dir('data/pet_images/Cat/*.jpg')
check_image_dir('data/pet_images/Dog/*.jpg')

In [ ]:
data_dir = 'data/pet_images'
batch_size = 64
ds_train = keras.preprocessing.image_dataset_from_directory(
    data_dir,
    validation_split = 0.2, 
    subset = 'training',
    seed = 13,
    image_size = (224, 224),
    batch_size = batch_size
)

ds_test = keras.preprocessing.image_dataset_from_directory(
    data_dir, 
    validation_split = 0.2, 
    subset = 'validation',
    seed = 13,
    image_size = (224, 224),
    batch_size = batch_size
)

In [ ]:
for x, y in ds_train:
    print(f'Training batch shape: features = {x.shape}, labels = {y.shape}')
    x_sample, y_sample = x, y

    break

### Pre-trained models
Many of image classification models are available inside `keras.applications` namespace. Let's see how simplest VGG-16 model can be loaded and used.

In [ ]:
vgg = keras.applications.VGG16(weights='imagenet')
inp = keras.applications.vgg16.preprocess_input(x_sample[:1])

res = vgg(inp)

print(f"Most probable class = {tf.argmax(res, 1)}")

keras.applications.vgg16.decode_predictions(res.numpy())

In [ ]:
vgg.summary()

### GPU Computations

In [ ]:
tf.config.list_physical_devices('GPU')

### Extracting VGG features

In [ ]:
vgg = keras.applications.VGG16(include_top = False)

inp = keras.applications.vgg16.preprocess_input(x_sample[:1])
res = vgg(inp)

print(f"Shape after applying VGG16: {res[0].shape}")

In [ ]:
num = batch_size * 50

ds_feature_train = ds_train.take(50).map(lambda x, y: (vgg(x), y))
ds_feature_test = ds_test.take(10).map(lambda x, y: (vgg(x), y))

for x, y in ds_feature_train:
    print(x.shape, y.shape)
    break

In [ ]:
model = keras.models.Sequential([
    keras.applications.VGG16(include_top = False, input_shape = (224, 224, 3)),
    keras.layers.Flatten(),
    keras.layers.Dense(1, activation = 'sigmoid')
])

model.layers[0].trainable = False
model.summary()

In [ ]:
model.compile(optimizer = 'adam', loss = 'binary_crossentropy', metrics = ['acc'])
hist = model.fit(ds_train, validation_data = ds_test)

### Saving and Loading the Model

In [ ]:
model.save('data/cats_dogs.tf')

model = keras.models.load_model('data/cats_dogs.tf')
model.layers[0].summary()

We can unfreeze all layers of conv base:

In [ ]:
model.layers[0].trainable = True

Or we can unfreeze just a few final layers:

In [ ]:
for i in range(len(model.layers[0].layers) - 4):
    model.layers[0].layers[i].trainable = False

model.summary()